In [1]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from torch.nn.functional import pad
from datasets import load_dataset , Dataset
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score


# Step 2: Define Configuration
device = "cuda"
model_name_or_path = 'bert-base-multilingual-uncased'
tokenizer_name_or_path = 'bert-base-multilingual-uncased'
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # Correct task type for sequence classification
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",  # Changed to a more specific prompt
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128  # Increased max_length to accommodate longer texts
lr = 3e-2
num_epochs = 50
batch_size = 8

# Step 3: Load Dataset

# dataset = pd.read_csv()

# dataset.iloc[0]

column_names = [label_column,"blank" ,text_column]
# Load the dataset without headers and assign the column names
dataset = pd.read_csv("/content/drive/MyDrive/societal_2l_train_bn.csv", header=None, names=column_names).head(500) #examples in dataset

dataset.drop("blank", axis=1, inplace=True)
dataset[label_column] = dataset[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset.head())
# Extract the specific range
#specific_range_dataset = dataset.iloc[1000:1201]  # Adjusted to include index 1200
#print(specific_range_dataset.head())
from sklearn.model_selection import train_test_split

# Split the preprocessed dataset into training and testing sets (80% train, 20% test)
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

# Now you have train_dataset and test_dataset for training and testing respectively
print(train_dataset)
dataset_train = Dataset.from_pandas(train_dataset)
dataset_test = Dataset.from_pandas(test_dataset)

# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Step 5: Preprocess Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], truncation=True, max_length=max_length, padding="max_length")
    model_inputs["labels"] = examples[label_column]
    return model_inputs

# Step 6: Apply Preprocessing
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

# Step 7: DataLoaders
train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=False, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)

# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)  # Binary classification
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())

# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    true_labels = []  # Initialize a list to collect true labels
    for step, batch in enumerate(tqdm(test_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())
        true_labels.extend(batch["labels"].detach().cpu().numpy())  # Collect true labels

    eval_epoch_loss = eval_loss / len(test_dataloader)
    train_epoch_loss = total_loss / len(train_dataloader)
        # Calculate evaluation metrics
    accuracy = accuracy_score(true_labels, eval_preds)
    f1 = f1_score(true_labels, eval_preds, average='weighted')
    recall = recall_score(true_labels, eval_preds, average='weighted')
    precision = precision_score(true_labels, eval_preds, average='weighted')


    print(f"Epoch {epoch}: train_loss={train_epoch_loss}, eval_loss={eval_epoch_loss}, accuracy={accuracy}, f1_score={f1}, recall={recall}, precision={precision}")


# After training, output the predictions
# Convert predictions and true labels to a DataFrame for easy analysis
results_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': eval_preds
})
print(results_df)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 25.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 75.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 16.1.0 which

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

Epoch 47: train_loss=0.36200839281082153, eval_loss=0.7061289548873901, accuracy=0.68, f1_score=0.6706666666666666, recall=0.68, precision=0.6898778833107191


100%|██████████| 13/13 [00:00<00:00, 14.36it/s]


Epoch 48: train_loss=0.3362278640270233, eval_loss=0.694098711013794, accuracy=0.72, f1_score=0.7168013055895551, recall=0.72, precision=0.7230937368642287


100%|██████████| 13/13 [00:00<00:00, 14.36it/s]

Epoch 49: train_loss=0.34283342957496643, eval_loss=0.6979077458381653, accuracy=0.7, f1_score=0.6950738916256156, recall=0.7, precision=0.7047190047190048
    true_label  predicted_label
0            1                1
1            0                1
2            0                1
3            1                1
4            0                1
..         ...              ...
95           1                0
96           1                0
97           1                1
98           1                1
99           1                1

[100 rows x 2 columns]


In [2]:
# Save the results to a CSV file
results_df.to_csv("/content/drive/MyDrive/model_predictions.csv", index=False)
print("Predictions saved to /content/drive/MyDrive/model_predictions.csv")

# Or print the first few rows of the results
print(results_df.head())

Predictions saved to /content/drive/MyDrive/model_predictions.csv
   true_label  predicted_label
0           1                1
1           0                1
2           0                1
3           1                1
4           0                1


In [3]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [4]:
accuracy = accuracy_score(true_labels, eval_preds)
f1 = f1_score(true_labels, eval_preds, average='weighted')
recall = recall_score(true_labels, eval_preds, average='weighted')
precision = precision_score(true_labels, eval_preds, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.7
F1 Score: 0.6950738916256156
Recall: 0.7
Precision: 0.7047190047190048


In [7]:
inputs = tokenizer(
    f'{text_column} : {"কংগ্রেস পার্টি #GST বাস্তবায়নে পূর্ণ সমর্থন দিয়েছে: বীরাপ্পা মইলিলাইভ এখানে আপডেট:"} Label : ',
    return_tensors="pt",
)

In [8]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    print(predictions.detach().cpu().numpy())

[1]


In [11]:
inputs = tokenizer(
    f'{text_column} : {"প্রথমে #পাঠানকোট তারপর #উরি এখন #বরামুল্লা"} Label : ',
    return_tensors="pt",
)

In [12]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    print(predictions.detach().cpu().numpy())

[0]


In [15]:
inputs = tokenizer(
    f'{text_column} : {"বেঙ্গালুরু: পাঠানকোট বীরের আত্মীয় দুঃখিত কারণ বাড়ির অংশটি @IndianExpress এর মাধ্যমে ধ্বংসের তালিকার URL-এর অধীনে আসে"} Label : ',
    return_tensors="pt",
)

In [16]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    print(predictions.detach().cpu().numpy())

[0]


In [27]:
inputs = tokenizer(
    f'{text_column} : {"প্রিয় ব্যবসায়ীরা, আপনার সুবিধার জন্য এবং আপনাকে এক ধাপ এগিয়ে রাখতে আমরা নিফটি সাপ্তাহিক নিউজলেটার এবং GST... URL আপডেট করেছি"} Label : ',
    return_tensors="pt",
)

In [28]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    print(predictions.detach().cpu().numpy())

[1]


In [31]:
inputs = tokenizer(
    f'{text_column} : {"জিএসটি মানে রূপান্তরের দিকে বড় পদক্ষেপ: লোকসভায় প্রধানমন্ত্রী মোদি: @ndtv-এর মাধ্যমে URL"} Label : ',
    return_tensors="pt",
)

In [32]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    print(predictions.detach().cpu().numpy())

[1]
